# **GAPFILLING DE MODELOS METABÓLICOS A ESCALA GENÓMICA USANDO DNNGIOR**

Tutorial adaptado de Boer et al. 2024 (DOI:10.1016/j.isci.2024.111349) por Maria Carolina Sisco. 

Improving genome-scale metabolic models of incomplete genomes with deep learning
Boer et al. 2024. DOI: 10.1016/j.isci.2024.111349]

- DNNgior: Deep Neural Network Guided Imputation of Reactomes
- GSMM: Genome Scale Metabolic Model 
- DNNgior utiliza IA para mejorar el proceso de gap-filling al aprender de la presencia y ausencia de reacciones metabólicas en diversos genomas bacterianos. 

### **INSTALACIÓN** 

Abra una terminal o un prompt de comando y digite el siguiente comando para crear un ambiente:


conda create --name dnngior python=3.10.16

Cuando se le solicite, confirme la instalación escribiendo "y" y pulsando Intro. Después, active el ambiente.

conda activate dnngior

1. Abra una terminal

2. Vaya al directorio raiz usando cd

3. Abra el archivo bashrc. Sugiero usar gedit, nano o vim

4. Adicione la siguiente línea en el bashrc:

export PATH=’path-to-gurobi-bin-folder/bin/:$PATH’

Para localizar la ruta a la carpeta Gurobi-bin, debe ir al directorio donde extrajo los archivos de la carpeta tar.gz descargada.

Segundo, necesitamos una licencia Gurobi. Puede obtener una "free academic named-user-license" aqui https://www.gurobi.com/features/academic-named-user-license/ con su email institucional. 

Clicke en named-user-license. Generará una grbgetkey para su máquina. Por favor, escriba (cambiando la X por su respectiva clave):

grbgetkey 9f4XXXX-XXXX-XXXX-XXXX-XXXXXXXXXX

Ahora, usted instalará DNNgior (dentro de su ambiente conda) con el siguiente comando:

pip install dnngior

Para ejecutar la pipeline de DNNGIOR en un jupyter notebook, necesita instalar jupyter dentro de su ambiente conda con el siguiente comando:

conda install -c conda-forge notebook -y<br>
o<br>
pip install notebook

Abra un nuevo notebook escribiendo jupyter-notebook en la terminal y haga un test de la instalación de dnngior digitando:

import dnngior

En algunos casos, puede encontrar una inconsistencia de versión con numpy, una de las dependencias de dnngior. Para solucionarlo, vaya a la terminal y escriba:

pip install numpy==1.23.5

Luego, vuelva a probar la instalación de dnngior. Debería ver esto:

Set parameter Username<br>
Set parameter LicenseID to value 2671523<br>
Academic license - for non-commercial use only - expires 2026-05-27<br>
WARNING: To enable the NN_Trainer script, you need to install<br>
tensorflow <https://www.tensorflow.org/install>,→<br>
The rest of dnngior features can be used without it.<br>

Ahora vamos a trabajar con DNNGIOR!

## **GAPFILLING USANDO UN MEDIO COMPLETO**
En este ejercicio vamos a realizar un gapfill (adicionar reacciones faltantes) a un modelo metabólico a escala genómica de Blautia, un género de bacteria anaeróbica con características probióticas. 

Vamos a explorar el modelo metabólico con algunos comandos básicos de Cobrapy!

In [ ]:
import cobra
from cobra.io import read_sbml_model
draft_reconstruction = read_sbml_model('bh_ungapfilled_model.sbml')

In [ ]:
draft_reconstruction.summary()

Note la función objetiva (biomass=tasa de crecimiento). También, no hay flujos.

In [ ]:
draft_reconstruction.optimize()

In [ ]:
draft_reconstruction.medium

Las reacciones exchange (uptake reactions-reacciones de absorción) están establecidas en un valor ilimitado, no hay ninguna restricción sobre lo que la bacteria puede tomar del medio, pero aún así, nuestro modelo no puede simular el crecimiento.

Empezemos el gapfilling de nuestro modelo!<br>
Importa la libreria dnngior y usa la clase Gapfill para adicionar las reacciones faltantes al modelo.

In [ ]:
import os, sys
path_to_blautia_model = ("bh_ungapfilled_model.sbml")

In [ ]:
import dnngior
gapfilled_model_complete = dnngior.Gapfill(draftModel = path_to_blautia_model, 
                                          medium = None, 
                                          objectiveName = 'bio1')

Haz un nuevo objeto del modelo con gapfill

In [ ]:
gf_model_compl_med = gapfilled_model_complete.gapfilledModel.copy()

In [ ]:
gf_model_compl_med.optimize()

### Está cresciendo!, la tasa de crecimiento después de la optimización es de 146.138 mmol/gDW/ hr (Milimoles por gramo de peso seco por hora), las unidades de flujo predeterminadas utilizadas en FBA.

Ahora veamos cuántas y cuáles reacciones agregó DNNgior para simular el crecimiento.

In [ ]:
print("Number of reactions added:", len(gapfilled_model_complete.added_reactions))
print("~~")
for reaction in gapfilled_model_complete.added_reactions:
    print(gf_model_compl_med.reactions.get_by_id(reaction).name)

In [ ]:
gf_model_compl_med.reactions.get_by_id('EX_cpd15432_e0')

In [ ]:
gf_model_compl_med.reactions.get_by_id('EX_cpd15511_e0')

## **GAPFILLING USANDO UN MEDIO DEFINIDO**

Primero, carga el archivo conteniendo los componentes del medio.

In [ ]:
medium_file_path = 'Nitrogen-Nitrite_media.tsv'

In [ ]:
import pandas as pd
new_medium = pd.read_csv(medium_file_path, sep="\t")
new_medium.head()

Vamos a hacer gapfilling de nuestro modelo metabólico para que pueda crecer en este medio. 

In [ ]:
gapfill_nitr = dnngior.Gapfill(path_to_blautia_model, medium_file = medium_file_path, objectiveName = 'bio1')

De nuevo, haga un nuevo objeto del modelo con gapfill y vea si está creciendo.


In [ ]:
gf_model_Nit_med = gapfill_nitr.gapfilledModel.copy()
gf_model_Nit_med.optimize()

Veamos cuántas y cuáles reacciones agregó DNNgior para simular el crecimiento en el medio de nitrito.

In [ ]:
print("Number of reactions added:", len(gapfill_nitr.added_reactions))
print("~~")
#for reaction in gapfill_nitr.added_reactions[:5]:
for reaction in gapfill_nitr.added_reactions:
    print(gf_model_Nit_med.reactions.get_by_id(reaction).name)